# inmemorycache 테스트

In [6]:
# 명시적 경로로 재로드
from dotenv import load_dotenv
load_dotenv(r"c:\Users\Admin\hipython\llm\.env", override=True)
from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

set_llm_cache(InMemoryCache())
llm = ChatOpenAI(model="gpt-4o-mini")

message = [HumanMessage(content="서울 광장시장에서 가장 맛있는 길거리 음식은?")]

In [4]:
%%time
# 첫 번째 호출 - API 실제 호출
response = llm.invoke(message)
print(response.content)
# Wall time: 약 2~3초

서울 광장시장에서 맛있는 길거리 음식으로는 다음과 같은 것들이 있습니다:

1. **떡볶이**: 매콤한 고추장 소스와 쫄깃한 떡이 어우러져 많은 사랑을 받습니다.
2. **순대**: 돼지의 창자를 이용한 순대는 다양한 소스와 함께 먹으면 더욱 맛있습니다.
3. **호떡**: 달콤한 시럽이 들어간 부드러운 팬케이크로, 간식으로 인기가 많습니다.
4. **튀김**: 각종 해물과 야채 튀김은 바삭하고 고소한 맛이 일품입니다.
5. **만두**: 찐만두나 군만두도 인기가 높아서 간편하게 즐길 수 있습니다.

광장시장은 다양한 먹거리가 풍부하니, 여러 가지 음식을 골고루 즐겨보는 것도 좋은 방법입니다!
CPU times: total: 15.6 ms
Wall time: 4.83 s


In [5]:
%%time
# 두 번째 호출 - 캐시에서 즉시 반환
response = llm.invoke(message)
print(response.content)
# Wall time: 약 1ms (거의 0)

서울 광장시장에서 맛있는 길거리 음식으로는 다음과 같은 것들이 있습니다:

1. **떡볶이**: 매콤한 고추장 소스와 쫄깃한 떡이 어우러져 많은 사랑을 받습니다.
2. **순대**: 돼지의 창자를 이용한 순대는 다양한 소스와 함께 먹으면 더욱 맛있습니다.
3. **호떡**: 달콤한 시럽이 들어간 부드러운 팬케이크로, 간식으로 인기가 많습니다.
4. **튀김**: 각종 해물과 야채 튀김은 바삭하고 고소한 맛이 일품입니다.
5. **만두**: 찐만두나 군만두도 인기가 높아서 간편하게 즐길 수 있습니다.

광장시장은 다양한 먹거리가 풍부하니, 여러 가지 음식을 골고루 즐겨보는 것도 좋은 방법입니다!
CPU times: total: 0 ns
Wall time: 1 ms


In [6]:
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

# 프로젝트 폴더에 .langchain.db 파일로 저장
set_llm_cache(SQLiteCache(database_path=".langchain.db"))

llm = ChatOpenAI(model="gpt-4o-mini")
message = [HumanMessage(content="코스피 지수란 무엇인가요? 한 문장으로 답해주세요.")]

In [7]:
%%time
# 첫 번째 호출 - API 실제 호출 후 파일에 저장
response = llm.invoke(message)
print(response.content)
# Wall time: 약 2~3초

코스피 지수는 한국 거래소에 상장된 대형 상장 주식들의 가격 변동을 나타내는 주식 시장의 대표적인 지표입니다.
CPU times: total: 15.6 ms
Wall time: 1.37 s


In [8]:
%%time
# 두 번째 호출 - 파일에서 즉시 반환 (프로그램 재시작 후에도 동일)
response = llm.invoke(message)
print(response.content)
# Wall time: 약 2~5ms

코스피 지수는 한국 거래소에 상장된 대형 상장 주식들의 가격 변동을 나타내는 주식 시장의 대표적인 지표입니다.
CPU times: total: 0 ns
Wall time: 1.49 ms


일반 캐시 (정확 일치)
─────────────────────────────────────────
"서울에서 맛있는 길거리 음식" → 캐시 적중
"서울에서 유명한 길거리 음식" → 캐시 미적중 (다른 문자열)

시맨틱 캐시 (의미 유사)
─────────────────────────────────────────
"서울에서 맛있는 길거리 음식" → 캐시 저장
"서울에서 유명한 길거리 음식" → 캐시 적중 (유사도 임계값 초과)

# Redis 활용하기

In [1]:
from dotenv import load_dotenv
import os
load_dotenv(r"c:\Users\Admin\hipython\llm\.env", override=True)

from langchain_core.globals import set_llm_cache
from langchain_redis import RedisSemanticCache          # 변경
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage

REDIS_URL = os.environ['REDIS_URL']



In [45]:
semantic_cache = RedisSemanticCache(
    redis_url=REDIS_URL,
    embeddings=OpenAIEmbeddings(model="text-embedding-3-small"),  # 파라미터명 변경
    distance_threshold=0.005,                                       # 파라미터명 변경
    ttl= 3600
)

In [46]:
set_llm_cache(semantic_cache)
llm = ChatOpenAI(model="gpt-4o-mini")
print("RedisSemanticCache 연결 완료")

RedisSemanticCache 연결 완료


In [51]:
%%time
response = llm.invoke([HumanMessage(content="코스피 지수란 무엇인가요? 한 문장으로 답해주세요.")])
print(response.content)

코스피 지수는 한국 증권거래소에서 상장된 주식들의 시가총액을 기준으로 산출되는 대한민국의 대표 주가 지수입니다.
CPU times: total: 0 ns
Wall time: 153 ms


In [52]:
%%time
response = llm.invoke([HumanMessage(content="코스피 지수란 무엇인가요? 한 문장으로 답해주세요.")])
print(response.content)

코스피 지수는 한국 증권거래소에서 상장된 주식들의 시가총액을 기준으로 산출되는 대한민국의 대표 주가 지수입니다.
CPU times: total: 0 ns
Wall time: 132 ms


In [53]:
%%time
response = llm.invoke([HumanMessage(content="코스피가 뭔지 간단히 설명해줘.")])
print(response.content)

코스피 지수는 한국 증권거래소에서 상장된 주식들의 시가총액을 기준으로 산출되는 대한민국의 대표 주가 지수입니다.
CPU times: total: 15.6 ms
Wall time: 287 ms


In [54]:
%%time
response = llm.invoke([HumanMessage(content="서울 광장시장에서 가장 맛있는 길거리 음식은?")])
print(response.content)

서울 광장시장에서 가장 유명한 길거리 음식 중 하나는 떡볶이입니다. 매콤하고 달콤한 소스에 쌀 떡과 어묵이 들어가 있어 많은 사람들이 사랑하는 메뉴입니다. 그 외에도 순대, 핫바, 튀김, 김밥 등 다양한 길거리 음식이 인기입니다. 특히, 광장시장은 전통적인 맛과 현대적인 퓨전 메뉴가 잘 어우러져 있어 다양한 선택을 즐길 수 있는 곳입니다. 방문하시면 꼭 여러 가지 음식을 시도해 보세요!
CPU times: total: 62.5 ms
Wall time: 3.51 s


In [55]:
%%time
response = llm.invoke([HumanMessage(content="코스닥이 뭔지 간단히 설명해줘.")])
print(response.content)

코스피 지수는 한국 증권거래소에서 상장된 주식들의 시가총액을 기준으로 산출되는 대한민국의 대표 주가 지수입니다.
CPU times: total: 0 ns
Wall time: 323 ms
